In [1]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report,confusion_matrix

I0000 00:00:1794233836.786957    5548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794233837.770199    5548 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794233841.253404    5548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
trainpath = "../Datasets/data/train/Medical/MRI/Brain Tumor/Training"
validpath = "../Datasets/data/vaild/Medical/MRI/Brain Tumor/Validation"
tespath = "../Datasets/data/test/Medical/MRI/Brain Tumor/Testing"

In [3]:
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_datagen =ImageDataGenerator(
    preprocessing_function = preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

In [4]:
training_set = train_datagen.flow_from_directory(
    trainpath,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary"
)

validation_set = valid_datagen.flow_from_directory(
    validpath,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary"
)

testing_set = test_datagen.flow_from_directory(
    tespath,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary",
    shuffle = False
    )

Found 7883 images belonging to 2 classes.


Found 1600 images belonging to 2 classes.
Found 781 images belonging to 2 classes.


In [5]:
training_set.class_indices

{'abnormal': 0, 'notumor': 1}

In [6]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False, input_shape=(224,224,3))

I0000 00:00:1794233852.257382    5548 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [8]:
base_model.trainable = False

In [9]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1,activation="sigmoid")(x)
model = Model(inputs=base_model.input,outputs=output)

In [83]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [84]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [85]:
class_weight ={0:1,1:2.2}

In [94]:
history = model.fit(training_set,validation_data=validation_set,epochs=10, callbacks=[early_stop],class_weight=class_weight)

Epoch 1/10


I0000 00:00:1790096612.615589   13924 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_268722__.262


477/493 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step - accuracy: 0.9845 - loss: 0.0588

I0000 00:00:1790096685.259950   13926 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_268722__.262


493/493 ━━━━━━━━━━━━━━━━━━━━ 96s 157ms/step - accuracy: 0.9841 - loss: 0.0621 - val_accuracy: 0.9600 - val_loss: 0.1125
Epoch 2/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 66s 134ms/step - accuracy: 0.9877 - loss: 0.0530 - val_accuracy: 0.9719 - val_loss: 0.0875
Epoch 3/10
493/493 ━━━━━━━━━━━━━━━━━━━━ 67s 135ms/step - accuracy: 0.9902 - loss: 0.0382 - val_accuracy: 0.9744 - val_loss: 0.0826


In [95]:
loss, accuracy = model.evaluate(testing_set)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9898 - loss: 0.0349
Test Loss: 0.03487157076597214
Test Accuracy: 0.98975670337677


In [96]:
predictions = model.predict(testing_set)

49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step


In [97]:
import numpy as np

y_pred = (predictions >= 0.5).astype(int)

In [98]:
y_true = testing_set.classes

In [ ]:
print(classification_report(y_true,y_pred,))
print(confusion_matrix(y_true, y_pred,))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       600
           1       0.99      0.97      0.98       181

    accuracy                           0.99       781
   macro avg       0.99      0.98      0.99       781
weighted avg       0.99      0.99      0.99       781

[[598   2]
 [  6 175]]


In [101]:
model.save("Brain_tumor_model.keras")